
# FCI graph-level benchmark: modified mixed-data KCI vs. standard KCI + one-hot

This notebook compares the **FCI structure-learning results** obtained with:

1. `RUN_MODE = "modified"`: revised mixed-data KCI on raw mixed variables, using `is_discrete`.
2. `RUN_MODE = "onehot"`: standard upstream KCI after one-hot encoding the discrete variables.

Run this notebook twice in the same folder:

- once with the **modified** causal-learn environment/kernel and `RUN_MODE = "modified"`;
- once with the **upstream py-why/causal-learn** environment/kernel and `RUN_MODE = "onehot"`.

The notebook saves raw bootstrap metrics and, once both method files exist, produces a combined comparison table.


In [1]:
from pathlib import Path
import contextlib
import io
import inspect
import platform
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

warnings.filterwarnings("ignore")
np.set_printoptions(precision=3, suppress=True)

# =====================
# User settings
# =====================
RUN_MODE = "onehot"      # "modified" in the modified environment; "onehot" in upstream py-why/causal-learn.

N_SAMPLES = [1000, 2000]
N_REPS = 50

ALPHA = 0.05
WIDTH_SETTINGS = ["empirical"]  # Add "median" only if you want FCI-level bandwidth sensitivity.
BASE_SEED = 20260410

# Concise default: two representative FCI settings.
SCENARIOS_TO_RUN = [
    "linear_group1_mixed_fci",
    "nonlinear_groups12_mixed_fci",
]

# To run all four FCI settings, uncomment this line after checking runtime:
# SCENARIOS_TO_RUN = ["linear_group1_mixed_fci", "linear_groups12_mixed_fci", "nonlinear_group1_mixed_fci", "nonlinear_groups12_mixed_fci"]

# Truth graph for scoring:
# - "observed_directed": score only the observed directed edges.
# - "pag_with_latent": also include bidirected edges induced by the two latent common causes.
EVAL_TRUTH_MODE = "pag_with_latent"

# For the one-hot baseline, prevent meaningless edges among dummy columns of the same original variable.
PROHIBIT_WITHIN_DUMMY_EDGES = True

OUT_DIR = Path("results_fci_graph_benchmark")
OUT_DIR.mkdir(exist_ok=True)

# Use new filenames so that Monte Carlo results are not mixed with earlier bootstrap-based files.
RAW_OUT = OUT_DIR / f"fci_graph_benchmark_{RUN_MODE}_mc.csv"
SUMMARY_OUT = OUT_DIR / f"fci_graph_benchmark_summary_{RUN_MODE}_mc.csv"

print("Python:", platform.python_version())
print("RUN_MODE:", RUN_MODE)
print("N_SAMPLES:", N_SAMPLES)
print("N_REPS:", N_REPS)
print("WIDTH_SETTINGS:", WIDTH_SETTINGS)
print("SCENARIOS_TO_RUN:", SCENARIOS_TO_RUN)
print("EVAL_TRUTH_MODE:", EVAL_TRUTH_MODE)
print("OUT_DIR:", OUT_DIR.resolve())


Python: 3.12.3
RUN_MODE: onehot
N_SAMPLES: [1000, 2000]
N_REPS: 50
WIDTH_SETTINGS: ['empirical']
SCENARIOS_TO_RUN: ['linear_group1_mixed_fci', 'nonlinear_groups12_mixed_fci']
EVAL_TRUTH_MODE: pag_with_latent
OUT_DIR: /home/jiang/results_fci_graph_benchmark


In [2]:

# Check causal-learn and import FCI utilities.
import causallearn
from causallearn.search.ConstraintBased.FCI import fci
from causallearn.utils.PCUtils.BackgroundKnowledge import BackgroundKnowledge
from causallearn.graph.GraphNode import GraphNode
from causallearn.graph.GeneralGraph import GeneralGraph
from causallearn.graph.Edge import Edge
from causallearn.graph.Endpoint import Endpoint
from causallearn.graph.AdjacencyConfusion import AdjacencyConfusion
from causallearn.graph.ArrowConfusion import ArrowConfusion

print("causallearn loaded from:", causallearn.__file__)

# Quick sanity check for the modified fork.
try:
    from causallearn.utils import cit as cit_module
    source = inspect.getsource(cit_module.KCI)
    print("KCI source contains 'is_discrete':", "is_discrete" in source)
except Exception as e:
    print("Could not inspect KCI source:", repr(e))


causallearn loaded from: /opt/jupyterhub-venv/lib/python3.12/site-packages/causallearn/__init__.py
KCI source contains 'is_discrete': False



## Scenario definitions



### Structural equations used in the representative FCI benchmarks


In [3]:

sigmoid = lambda z: 1.0 / (1.0 + np.exp(-z))


def discretize_tertile(values):
    q1, q2 = np.quantile(values, [1.0 / 3.0, 2.0 / 3.0])
    return np.digitize(values, bins=[q1, q2])


def generate_linear_group1_mixed_fci(n, seed):
    rng = np.random.default_rng(seed)
    U1 = rng.normal(0, 1, n)
    U2 = rng.normal(0, 1, n)

    X1 = 0.9 * U1 + rng.normal(0, 1, n)
    B1 = rng.binomial(1, sigmoid(0.8 * U1), size=n)
    T1 = discretize_tertile(rng.normal(0, 1, n))

    X2_1 = 0.9 * X1 - 0.8 * B1 - 0.4 * U2 + rng.normal(0, 1, n)
    X2_2 = 0.4 * B1 - 0.3 * T1 + rng.normal(0, 1, n)
    X2_3 = 0.5 * T1 + 0.5 * U2 + rng.normal(0, 1, n)
    X2_4 = 0.3 * X1 + 0.6 * X2_1 + 0.7 * X2_3 + rng.normal(0, 1, n)
    B3 = rng.binomial(1, sigmoid(0.9 * X2_3 + 1.1 * X2_4), size=n)

    return pd.DataFrame({
        "X1": X1, "B1": B1, "T1": T1,
        "X2_1": X2_1, "X2_2": X2_2, "X2_3": X2_3, "X2_4": X2_4,
        "B3": B3,
    })


def generate_linear_groups12_mixed_fci(n, seed):
    rng = np.random.default_rng(seed)
    U1 = rng.normal(0, 1, n)
    U2 = rng.normal(0, 1, n)

    X1 = 0.9 * U1 + rng.normal(0, 1, n)
    B1 = rng.binomial(1, sigmoid(0.8 * U1), size=n)
    T1 = discretize_tertile(rng.normal(0, 1, n))

    X2_1 = 0.9 * B1 - 0.4 * T1 + rng.normal(0, 1, n)
    X2_2 = -0.7 * T1 - 0.4 * U2 + rng.normal(0, 1, n)
    B2 = rng.binomial(1, sigmoid(0.8 * X1 - 0.7 * B1 + 0.5 * U2), size=n)

    zt2 = 0.8 * X1 + 0.6 * X2_2 - 0.4 * B2 + 0.7 + rng.normal(0, 1, n)
    T2 = discretize_tertile(zt2)
    B3 = rng.binomial(1, sigmoid(0.7 * X2_2 + 0.5 * T2 + 0.2), size=n)

    return pd.DataFrame({
        "X1": X1, "B1": B1, "T1": T1,
        "B2": B2, "X2_1": X2_1, "X2_2": X2_2, "T2": T2,
        "B3": B3,
    })


def generate_nonlinear_group1_mixed_fci(n, seed):
    rng = np.random.default_rng(seed)
    U1 = rng.normal(0, 1, n)
    U2 = rng.normal(0, 1, n)

    X1 = 0.9 * np.sin(U1) + rng.normal(0, 1, n)
    B1 = rng.binomial(1, sigmoid(0.8 * (U1 ** 2)), size=n)
    T1 = discretize_tertile(rng.normal(0, 1, n))

    X2_1 = 0.9 * np.sin(X1) - 0.8 * (B1 ** 2) - 0.4 * (U2 ** 2) + rng.normal(0, 1, n)
    X2_2 = 0.4 * np.cos(B1) - 0.3 * (T1 ** 2) + rng.normal(0, 1, n)
    X2_3 = 0.5 * np.cos(1.3 * T1) + 0.5 * np.sin(U2) + rng.normal(0, 1, n)
    X2_4 = 0.3 * np.sin(X1) + 0.6 * np.log1p(np.abs(X2_1)) + 0.7 * np.tanh(0.4 * X2_3) + rng.normal(0, 1, n)
    B3 = rng.binomial(1, sigmoid(0.9 * np.sin(X2_3) + 1.1 * (X2_4 ** 2)), size=n)

    return pd.DataFrame({
        "X1": X1, "B1": B1, "T1": T1,
        "X2_1": X2_1, "X2_2": X2_2, "X2_3": X2_3, "X2_4": X2_4,
        "B3": B3,
    })


def generate_nonlinear_groups12_mixed_fci(n, seed):
    rng = np.random.default_rng(seed)
    U1 = rng.normal(0, 1, n)
    U2 = rng.normal(0, 1, n)

    X1 = 0.9 * np.sin(U1) + rng.normal(0, 1, n)
    B1 = rng.binomial(1, sigmoid(0.8 * (U1 ** 2)), size=n)
    T1 = discretize_tertile(rng.normal(0, 1, n))

    X2_1 = 0.9 * np.sin(B1) - 0.4 * (T1 ** 2) + rng.normal(0, 1, n)
    X2_2 = -0.7 * np.cos(T1) - 0.4 * (U2 ** 2) + rng.normal(0, 1, n)
    B2 = rng.binomial(1, sigmoid(0.8 * (X1 ** 2) - 0.7 * np.cos(1.3 * B1) + 0.5 * np.sin(U2)), size=n)

    zt2 = 0.8 * np.sin(X1) + 0.6 * np.log1p(np.abs(X2_2)) - 0.4 * np.tanh(0.7 * B2) + rng.normal(0, 1, n)
    T2 = discretize_tertile(zt2)
    B3 = rng.binomial(1, sigmoid(0.7 * np.sin(X2_2) + 0.5 * (T2 ** 2)), size=n)

    return pd.DataFrame({
        "X1": X1, "B1": B1, "T1": T1,
        "B2": B2, "X2_1": X2_1, "X2_2": X2_2, "T2": T2,
        "B3": B3,
    })


SCENARIO_LABELS = {
    "linear_group1_mixed_fci": "Linear: mixed variables only in group 1",
    "linear_groups12_mixed_fci": "Linear: mixed variables in groups 1 and 2",
    "nonlinear_group1_mixed_fci": "Nonlinear: mixed variables only in group 1",
    "nonlinear_groups12_mixed_fci": "Nonlinear: mixed variables in groups 1 and 2",
}

OBSERVED_EDGES_GROUP1 = [
    ("X1", "X2_1"), ("X1", "X2_4"),
    ("B1", "X2_1"), ("B1", "X2_2"),
    ("T1", "X2_2"), ("T1", "X2_3"),
    ("X2_1", "X2_4"), ("X2_3", "X2_4"),
    ("X2_3", "B3"), ("X2_4", "B3"),
]

OBSERVED_EDGES_GROUPS12 = [
    ("X1", "B2"), ("X1", "T2"),
    ("B1", "B2"), ("B1", "X2_1"),
    ("T1", "X2_1"), ("T1", "X2_2"),
    ("B2", "T2"), ("X2_2", "T2"),
    ("X2_2", "B3"), ("T2", "B3"),
]

SCENARIOS = {
    "linear_group1_mixed_fci": {
        "generator": generate_linear_group1_mixed_fci,
        "node_names": ["X1", "B1", "T1", "X2_1", "X2_2", "X2_3", "X2_4", "B3"],
        "is_discrete": np.array([False, True, True, False, False, False, False, True], dtype=bool),
        "categorical_levels": {"B1": [0, 1], "T1": [0, 1, 2], "B3": [0, 1]},
        "exog": ["X1", "B1", "T1"],
        "endog": ["X2_1", "X2_2", "X2_3", "X2_4"],
        "sink": ["B3"],
        "observed_edges": OBSERVED_EDGES_GROUP1,
        "latent_pairs": [("X1", "B1"), ("X2_1", "X2_3")],
    },
    "linear_groups12_mixed_fci": {
        "generator": generate_linear_groups12_mixed_fci,
        "node_names": ["X1", "B1", "T1", "B2", "X2_1", "X2_2", "T2", "B3"],
        "is_discrete": np.array([False, True, True, True, False, False, True, True], dtype=bool),
        "categorical_levels": {"B1": [0, 1], "T1": [0, 1, 2], "B2": [0, 1], "T2": [0, 1, 2], "B3": [0, 1]},
        "exog": ["X1", "B1", "T1"],
        "endog": ["B2", "X2_1", "X2_2", "T2"],
        "sink": ["B3"],
        "observed_edges": OBSERVED_EDGES_GROUPS12,
        "latent_pairs": [("X1", "B1"), ("B2", "X2_2")],
    },
    "nonlinear_group1_mixed_fci": {
        "generator": generate_nonlinear_group1_mixed_fci,
        "node_names": ["X1", "B1", "T1", "X2_1", "X2_2", "X2_3", "X2_4", "B3"],
        "is_discrete": np.array([False, True, True, False, False, False, False, True], dtype=bool),
        "categorical_levels": {"B1": [0, 1], "T1": [0, 1, 2], "B3": [0, 1]},
        "exog": ["X1", "B1", "T1"],
        "endog": ["X2_1", "X2_2", "X2_3", "X2_4"],
        "sink": ["B3"],
        "observed_edges": OBSERVED_EDGES_GROUP1,
        "latent_pairs": [("X1", "B1"), ("X2_1", "X2_3")],
    },
    "nonlinear_groups12_mixed_fci": {
        "generator": generate_nonlinear_groups12_mixed_fci,
        "node_names": ["X1", "B1", "T1", "B2", "X2_1", "X2_2", "T2", "B3"],
        "is_discrete": np.array([False, True, True, True, False, False, True, True], dtype=bool),
        "categorical_levels": {"B1": [0, 1], "T1": [0, 1, 2], "B2": [0, 1], "T2": [0, 1, 2], "B3": [0, 1]},
        "exog": ["X1", "B1", "T1"],
        "endog": ["B2", "X2_1", "X2_2", "T2"],
        "sink": ["B3"],
        "observed_edges": OBSERVED_EDGES_GROUPS12,
        "latent_pairs": [("X1", "B1"), ("B2", "X2_2")],
    },
}

for name in SCENARIOS_TO_RUN:
    spec = SCENARIOS[name]
    print(name, "variables:", spec["node_names"])


linear_group1_mixed_fci variables: ['X1', 'B1', 'T1', 'X2_1', 'X2_2', 'X2_3', 'X2_4', 'B3']
nonlinear_groups12_mixed_fci variables: ['X1', 'B1', 'T1', 'B2', 'X2_1', 'X2_2', 'T2', 'B3']


### Collapsing the one-hot baseline back to the original-variable PAG



In [4]:

TAIL, ARROW, CIRCLE, NULL = -1, 1, 2, 0


def onehot_encode_with_fixed_levels(df, categorical_levels):
    parts = []
    expanded_to_original = {}
    for col in df.columns:
        if col in categorical_levels:
            cat = pd.Categorical(df[col], categories=categorical_levels[col])
            dummies = pd.get_dummies(cat, prefix=col, prefix_sep="__", dtype=float)
            parts.append(dummies)
            for dummy_col in dummies.columns:
                expanded_to_original[dummy_col] = col
        else:
            parts.append(df[[col]].astype(float))
            expanded_to_original[col] = col
    encoded = pd.concat(parts, axis=1)
    return encoded, expanded_to_original


def expand_names(var_names, expanded_to_original):
    return [col for col, original in expanded_to_original.items() if original in set(var_names)]


def build_background_knowledge(node_names, exog_names, endog_names, sink_names):
    nodes = {name: GraphNode(name) for name in node_names}
    bk = BackgroundKnowledge()

    # Prohibit Endogenous -> Exogenous.
    for endog in endog_names:
        for exog in exog_names:
            bk.add_forbidden_by_node(nodes[endog], nodes[exog])

    # Prohibit Exogenous <-> Sink.
    for sink in sink_names:
        for exog in exog_names:
            bk.add_forbidden_by_node(nodes[sink], nodes[exog])
            bk.add_forbidden_by_node(nodes[exog], nodes[sink])

    # Prohibit Sink -> Endogenous.
    for sink in sink_names:
        for endog in endog_names:
            bk.add_forbidden_by_node(nodes[sink], nodes[endog])

    return bk, nodes


def add_within_dummy_forbidden_edges(bk, nodes, expanded_to_original):
    original_to_expanded = {}
    for col, original in expanded_to_original.items():
        original_to_expanded.setdefault(original, []).append(col)
    for original, cols in original_to_expanded.items():
        if len(cols) <= 1:
            continue
        for i, a in enumerate(cols):
            for b in cols[i + 1:]:
                bk.add_forbidden_by_node(nodes[a], nodes[b])
                bk.add_forbidden_by_node(nodes[b], nodes[a])


def collapse_endpoint(endpoints):
    endpoints = [int(v) for v in endpoints if int(v) != NULL]
    if not endpoints:
        return NULL
    unique = set(endpoints)
    if len(unique) == 1:
        return endpoints[0]
    # Conservative endpoint if dummy-level edges disagree.
    return CIRCLE


def collapse_pag_matrix(adjacency_matrix_expanded, expanded_names, original_names, expanded_to_original):
    n_original = len(original_names)
    collapsed = np.zeros((n_original, n_original), dtype=int)
    for i, left in enumerate(original_names):
        for j in range(i + 1, n_original):
            right = original_names[j]
            left_endpoints = []
            right_endpoints = []
            for a, col_a in enumerate(expanded_names):
                if expanded_to_original[col_a] != left:
                    continue
                for b, col_b in enumerate(expanded_names):
                    if expanded_to_original[col_b] != right:
                        continue
                    if adjacency_matrix_expanded[a, b] != NULL or adjacency_matrix_expanded[b, a] != NULL:
                        left_endpoints.append(adjacency_matrix_expanded[a, b])
                        right_endpoints.append(adjacency_matrix_expanded[b, a])
            collapsed[i, j] = collapse_endpoint(left_endpoints)
            collapsed[j, i] = collapse_endpoint(right_endpoints)
    return collapsed


def adjacency_matrix_to_general_graph(adjacency_matrix, node_names):
    endpoint_map = {
        TAIL: Endpoint.TAIL,
        ARROW: Endpoint.ARROW,
        CIRCLE: Endpoint.CIRCLE,
    }
    nodes = [GraphNode(name) for name in node_names]
    graph = GeneralGraph(nodes)
    n = len(node_names)
    for i in range(n):
        for j in range(i + 1, n):
            left = int(adjacency_matrix[i, j])
            right = int(adjacency_matrix[j, i])
            if left == NULL and right == NULL:
                continue
            graph.add_edge(Edge(nodes[i], nodes[j], endpoint_map[left], endpoint_map[right]))
    return graph


def build_true_graph(spec, truth_mode="pag_with_latent"):
    node_names = spec["node_names"]
    nodes = {name: GraphNode(name) for name in node_names}
    graph = GeneralGraph([nodes[name] for name in node_names])

    for src, dst in spec["observed_edges"]:
        graph.add_directed_edge(nodes[src], nodes[dst])

    if truth_mode == "pag_with_latent":
        for a, b in spec.get("latent_pairs", []):
            graph.add_edge(Edge(nodes[a], nodes[b], Endpoint.ARROW, Endpoint.ARROW))
    elif truth_mode != "observed_directed":
        raise ValueError(f"Unknown truth_mode: {truth_mode}")

    return graph


def prepare_representation(df, spec, run_mode):
    original_names = spec["node_names"]
    if run_mode == "modified":
        data = df[original_names].values.astype(float)
        run_names = original_names
        expanded_to_original = {name: name for name in original_names}
        exog = spec["exog"]
        endog = spec["endog"]
        sink = spec["sink"]
        is_discrete = spec["is_discrete"]
        bk, nodes = build_background_knowledge(run_names, exog, endog, sink)
        return data, run_names, expanded_to_original, bk, is_discrete

    if run_mode == "onehot":
        encoded, expanded_to_original = onehot_encode_with_fixed_levels(df[original_names], spec["categorical_levels"])
        run_names = list(encoded.columns)
        exog = expand_names(spec["exog"], expanded_to_original)
        endog = expand_names(spec["endog"], expanded_to_original)
        sink = expand_names(spec["sink"], expanded_to_original)
        bk, nodes = build_background_knowledge(run_names, exog, endog, sink)
        if PROHIBIT_WITHIN_DUMMY_EDGES:
            add_within_dummy_forbidden_edges(bk, nodes, expanded_to_original)
        return encoded.values.astype(float), run_names, expanded_to_original, bk, None

    raise ValueError("RUN_MODE must be either 'modified' or 'onehot'.")


## Run FCI and compute graph-recovery metrics


In [5]:
def call_fci_safely(data, node_names, background_knowledge, run_mode, is_discrete, est_width):
    kwargs = {
        "independence_test_method": "kci",
        "alpha": ALPHA,
        "node_names": node_names,
        "background_knowledge": background_knowledge,
        "show_progress": False,
        "verbose": False,
    }
    if est_width is not None:
        kwargs["est_width"] = est_width
    if run_mode == "modified":
        kwargs["is_discrete"] = is_discrete

    try:
        return fci(data, **kwargs)[0]
    except TypeError as e:
        # Some causal-learn versions may not forward est_width through fci().
        # If that is the only issue, retry without est_width.
        if est_width is not None and "est_width" in str(e):
            kwargs.pop("est_width", None)
            return fci(data, **kwargs)[0]
        raise


def f1_score(precision, recall):
    if precision is None or recall is None:
        return np.nan
    if pd.isna(precision) or pd.isna(recall):
        return np.nan
    return 0.0 if (precision + recall) == 0 else 2 * precision * recall / (precision + recall)


def compute_metrics(true_graph, estimated_graph):
    adj_conf = AdjacencyConfusion(true_graph, estimated_graph)
    arrow_conf = ArrowConfusion(true_graph, estimated_graph)

    precision_skeleton = adj_conf.get_adj_precision()
    recall_skeleton = adj_conf.get_adj_recall()
    precision_arrow = arrow_conf.get_arrows_precision()
    recall_arrow = arrow_conf.get_arrows_recall()

    return {
        "precision_skeleton": precision_skeleton,
        "recall_skeleton": recall_skeleton,
        "f1_skeleton": f1_score(precision_skeleton, recall_skeleton),
        "precision_arrow": precision_arrow,
        "recall_arrow": recall_arrow,
        "f1_arrow": f1_score(precision_arrow, recall_arrow),
    }


def run_one_fci(df, spec, run_mode, est_width):
    data, run_names, expanded_to_original, bk, is_discrete = prepare_representation(df, spec, run_mode)

    start = time.perf_counter()
    with contextlib.redirect_stdout(io.StringIO()):
        graph_expanded = call_fci_safely(data, run_names, bk, run_mode, is_discrete, est_width)
    runtime_sec = time.perf_counter() - start

    if run_mode == "onehot":
        A_collapsed = collapse_pag_matrix(graph_expanded.graph, run_names, spec["node_names"], expanded_to_original)
        graph_original = adjacency_matrix_to_general_graph(A_collapsed, spec["node_names"])
        n_features = data.shape[1]
    else:
        graph_original = graph_expanded
        n_features = data.shape[1]

    true_graph = build_true_graph(spec, truth_mode=EVAL_TRUTH_MODE)
    metrics = compute_metrics(true_graph, graph_original)
    metrics.update({
        "runtime_sec": runtime_sec,
        "n_features": n_features,
        "n_original_variables": len(spec["node_names"]),
    })
    return metrics


def run_fci_benchmark():
    """Run an independent Monte Carlo FCI benchmark.

    This replaces the earlier bootstrap design. Each replication now calls the
    scenario generator with a new seed, so each row is based on an independently
    generated synthetic dataset.
    """
    rng = np.random.default_rng(BASE_SEED)
    rows = []

    for scenario_name in SCENARIOS_TO_RUN:
        spec = SCENARIOS[scenario_name]
        print(f"\n=== Scenario: {scenario_name} ===")

        for n_sample in N_SAMPLES:
            print(f"--- n = {n_sample} ---")
            sim_seeds = [int(rng.integers(0, 2**32 - 1)) for _ in range(N_REPS)]

            for est_width in WIDTH_SETTINGS:
                for rep, sim_seed in enumerate(sim_seeds, start=1):
                    sim_df = spec["generator"](n_sample, sim_seed)
                    row = {
                        "scenario": scenario_name,
                        "scenario_label": SCENARIO_LABELS[scenario_name],
                        "method": RUN_MODE,
                        "n": n_sample,
                        "rep": rep,
                        "dataset_seed": sim_seed,
                        "alpha": ALPHA,
                        "est_width": est_width,
                        "truth_mode": EVAL_TRUTH_MODE,
                        "resampling": "independent_monte_carlo",
                        "error": "",
                    }
                    try:
                        metrics = run_one_fci(sim_df, spec, RUN_MODE, est_width)
                        row.update(metrics)
                    except Exception as e:
                        row["error"] = repr(e)
                        row.update({
                            "precision_skeleton": np.nan,
                            "recall_skeleton": np.nan,
                            "f1_skeleton": np.nan,
                            "precision_arrow": np.nan,
                            "recall_arrow": np.nan,
                            "f1_arrow": np.nan,
                            "runtime_sec": np.nan,
                            "n_features": np.nan,
                            "n_original_variables": len(spec["node_names"]),
                        })
                    rows.append(row)
                    if rep == 1 or rep % 5 == 0 or rep == N_REPS:
                        status = "ok" if row["error"] == "" else "ERROR"
                        runtime = row.get("runtime_sec", np.nan)
                        runtime_text = f"{runtime:.2f}s" if np.isfinite(runtime) else "NA"
                        print(
                            f"  n={n_sample}, width={est_width}, "
                            f"rep={rep}/{N_REPS}, {status}, runtime={runtime_text}"
                        )

    return pd.DataFrame(rows)


In [6]:
%%time
raw_results = run_fci_benchmark()
raw_results.to_csv(RAW_OUT, index=False)
print("Saved raw results to:", RAW_OUT)
display(raw_results.head())
print("Failures:", (raw_results["error"] != "").sum())
if (raw_results["error"] != "").any():
    display(raw_results.loc[raw_results["error"] != "", ["scenario", "rep", "est_width", "error"]].head())



=== Scenario: linear_group1_mixed_fci ===
--- n = 1000 ---
  n=1000, width=empirical, rep=1/50, ok, runtime=107.68s
  n=1000, width=empirical, rep=5/50, ok, runtime=87.07s
  n=1000, width=empirical, rep=10/50, ok, runtime=108.95s
  n=1000, width=empirical, rep=15/50, ok, runtime=104.74s
  n=1000, width=empirical, rep=20/50, ok, runtime=89.67s
  n=1000, width=empirical, rep=25/50, ok, runtime=127.01s
  n=1000, width=empirical, rep=30/50, ok, runtime=143.11s
  n=1000, width=empirical, rep=35/50, ok, runtime=104.81s
  n=1000, width=empirical, rep=40/50, ok, runtime=97.13s
  n=1000, width=empirical, rep=45/50, ok, runtime=158.11s
  n=1000, width=empirical, rep=50/50, ok, runtime=82.43s
--- n = 2000 ---
  n=2000, width=empirical, rep=1/50, ok, runtime=503.21s
  n=2000, width=empirical, rep=5/50, ok, runtime=612.56s
  n=2000, width=empirical, rep=10/50, ok, runtime=410.57s
  n=2000, width=empirical, rep=15/50, ok, runtime=493.96s
  n=2000, width=empirical, rep=20/50, ok, runtime=650.78s
  n

,scenario,scenario_label,method,n,rep,dataset_seed,alpha,est_width,truth_mode,resampling,error,precision_skeleton,recall_skeleton,f1_skeleton,precision_arrow,recall_arrow,f1_arrow,runtime_sec,n_features,n_original_variables
0,linear_group1_mixed_fci,Linear: mixed variables only in group 1,onehot,1000,1,1142460215,0.05,empirical,pag_with_latent,independent_monte_carlo,,1.000000,0.5,0.666667,1.00,0.428571,0.600000,107.678329,12,8
1,linear_group1_mixed_fci,Linear: mixed variables only in group 1,onehot,1000,2,3244730521,0.05,empirical,pag_with_latent,independent_monte_carlo,,1.000000,0.5,0.666667,1.00,0.428571,0.600000,85.620996,12,8
2,linear_group1_mixed_fci,Linear: mixed variables only in group 1,onehot,1000,3,873013579,0.05,empirical,pag_with_latent,independent_monte_carlo,,1.000000,0.5,0.666667,1.00,0.428571,0.600000,91.997612,12,8
3,linear_group1_mixed_fci,Linear: mixed variables only in group 1,onehot,1000,4,4002622319,0.05,empirical,pag_with_latent,independent_monte_carlo,,1.000000,0.5,0.666667,1.00,0.428571,0.600000,101.398448,12,8
4,linear_group1_mixed_fci,Linear: mixed variables only in group 1,onehot,1000,5,1655443511,0.05,empirical,pag_with_latent,independent_monte_carlo,,0.857143,0.5,0.631579,0.75,0.428571,0.545455,87.065011,12,8


Failures: 0
CPU times: user 51d 8h 46min 18s, sys: 3h 27min 11s, total: 51d 12h 13min 30s
Wall time: 1d 2min 46s



## Summarize the current run


In [7]:
def _coerce_fci_numeric_columns(df):
    """Coerce metric columns back to numeric after reading CSV files.

    This is needed because pd.read_csv(..., keep_default_na=False) preserves
    empty strings in error rows, which can make metric columns object dtype.
    """
    out = df.copy()

    numeric_cols = [
        "n", "rep", "alpha",
        "precision_skeleton", "recall_skeleton", "f1_skeleton",
        "precision_arrow", "recall_arrow", "f1_arrow",
        "runtime_sec", "n_features", "n_original_variables",
    ]
    for col in numeric_cols:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")

    return out


def summarize_fci_results(df):
    df = _coerce_fci_numeric_columns(df)

    if "error" not in df.columns:
        df["error"] = ""

    valid = df[df["error"].fillna("").astype(str) == ""].copy()

    if "resampling" not in valid.columns:
        valid["resampling"] = "unspecified"

    metrics = [
        "precision_skeleton", "recall_skeleton", "f1_skeleton",
        "precision_arrow", "recall_arrow", "f1_arrow",
        "runtime_sec", "n_features",
    ]

    # Ensure all metric columns exist and are numeric even if a previous CSV
    # contained blanks or mixed types.
    for m in metrics:
        if m not in valid.columns:
            valid[m] = np.nan
        valid[m] = pd.to_numeric(valid[m], errors="coerce")

    group_cols = [
        "scenario", "scenario_label", "method", "n",
        "est_width", "truth_mode", "resampling",
    ]

    agg = valid.groupby(group_cols, as_index=False).agg(
        **{f"{m}_mean": (m, "mean") for m in metrics},
        **{f"{m}_sd": (m, "std") for m in metrics},
        n_success=("rep", "count"),
    )

    return agg.sort_values(["scenario", "method", "n", "est_width"])


if "raw_results" in globals():
    summary_current = summarize_fci_results(raw_results)
    summary_current.to_csv(SUMMARY_OUT, index=False)
    print("Saved summary to:", SUMMARY_OUT)
    display(summary_current)
else:
    print("raw_results is not defined. If you only want to compare existing CSV files, continue to the comparison cells below.")


Saved summary to: results_fci_graph_benchmark/fci_graph_benchmark_summary_onehot_mc.csv


,scenario,scenario_label,method,n,est_width,truth_mode,resampling,precision_skeleton_mean,recall_skeleton_mean,f1_skeleton_mean,...,n_features_mean,precision_skeleton_sd,recall_skeleton_sd,f1_skeleton_sd,precision_arrow_sd,recall_arrow_sd,f1_arrow_sd,runtime_sec_sd,n_features_sd,n_success
0,linear_group1_mixed_fci,Linear: mixed variables only in group 1,onehot,1000,empirical,pag_with_latent,independent_monte_carlo,0.980357,0.511667,0.670553,...,12.0,0.049235,0.050536,0.046014,0.085978,0.066130,0.063003,22.756833,0.0,50
1,linear_group1_mixed_fci,Linear: mixed variables only in group 1,onehot,2000,empirical,pag_with_latent,independent_monte_carlo,0.991429,0.500000,0.663571,...,12.0,0.034271,0.041239,0.037707,0.034271,0.058172,0.054779,105.937636,0.0,50
2,nonlinear_groups12_mixed_fci,Nonlinear: mixed variables in groups 1 and 2,onehot,1000,empirical,pag_with_latent,independent_monte_carlo,0.940000,0.198333,0.323308,...,15.0,0.135777,0.055558,0.080150,0.221308,0.065434,0.084845,34.789585,0.0,50
3,nonlinear_groups12_mixed_fci,Nonlinear: mixed variables in groups 1 and 2,onehot,2000,empirical,pag_with_latent,independent_monte_carlo,0.993333,0.153333,0.260366,...,15.0,0.047140,0.063799,0.095009,0.357176,0.085422,0.132601,198.664390,0.0,50



## Compare both methods after running the notebook twice

After running once with `RUN_MODE="modified"` and once with `RUN_MODE="onehot"`, rerun the following cells to create combined tables and figures.


In [8]:
def load_available_fci_results(out_dir):
    paths = [
        out_dir / "fci_graph_benchmark_modified_mc.csv",
        out_dir / "fci_graph_benchmark_onehot_mc.csv",
    ]
    frames = []
    for path in paths:
        if path.exists():
            frames.append(pd.read_csv(path, keep_default_na=False))
    if not frames:
        return None
    return pd.concat(frames, ignore_index=True)

all_results = load_available_fci_results(OUT_DIR)
if all_results is None:
    print("No FCI result files found yet.")
else:
    print("Methods found:", sorted(all_results["method"].dropna().unique().tolist()))
    print("Rows:", len(all_results))
    display(all_results.head())


Methods found: ['modified', 'onehot']
Rows: 400


,scenario,scenario_label,method,n,rep,dataset_seed,alpha,est_width,truth_mode,resampling,error,precision_skeleton,recall_skeleton,f1_skeleton,precision_arrow,recall_arrow,f1_arrow,runtime_sec,n_features,n_original_variables
0,linear_group1_mixed_fci,Linear: mixed variables only in group 1,modified,1000,1,1142460215,0.05,empirical,pag_with_latent,independent_monte_carlo,,1.0,0.916667,0.956522,0.909091,0.714286,0.8,77.419046,8,8
1,linear_group1_mixed_fci,Linear: mixed variables only in group 1,modified,1000,2,3244730521,0.05,empirical,pag_with_latent,independent_monte_carlo,,1.0,0.833333,0.909091,1.0,0.642857,0.782609,51.869059,8,8
2,linear_group1_mixed_fci,Linear: mixed variables only in group 1,modified,1000,3,873013579,0.05,empirical,pag_with_latent,independent_monte_carlo,,1.0,0.833333,0.909091,1.0,0.642857,0.782609,53.017832,8,8
3,linear_group1_mixed_fci,Linear: mixed variables only in group 1,modified,1000,4,4002622319,0.05,empirical,pag_with_latent,independent_monte_carlo,,1.0,0.750000,0.857143,1.0,0.500000,0.666667,60.941026,8,8
4,linear_group1_mixed_fci,Linear: mixed variables only in group 1,modified,1000,5,1655443511,0.05,empirical,pag_with_latent,independent_monte_carlo,,1.0,0.833333,0.909091,1.0,0.642857,0.782609,50.235975,8,8


In [9]:
METHOD_LABELS = {
    "modified": "Revised mixed-data KCI",
    "onehot": "Standard KCI + one-hot",
}

if all_results is None or not {"modified", "onehot"}.issubset(set(all_results["method"].dropna().unique())):
    print("Run once with RUN_MODE='modified' and once with RUN_MODE='onehot', then rerun this cell.")
else:
    summary_all = summarize_fci_results(all_results)
    summary_all["method_label"] = summary_all["method"].map(METHOD_LABELS)
    combined_summary_path = OUT_DIR / "fci_graph_benchmark_summary_combined_mc.csv"
    summary_all.to_csv(combined_summary_path, index=False)
    print("Saved combined summary to:", combined_summary_path)

    # Main comparison table for empirical width.
    main = summary_all[summary_all["est_width"] == "empirical"].copy()
    keep_cols = [
        "precision_skeleton_mean", "recall_skeleton_mean", "f1_skeleton_mean",
        "precision_arrow_mean", "recall_arrow_mean", "f1_arrow_mean",
        "runtime_sec_mean", "n_features_mean", "n_success",
    ]
    main_table = main.pivot_table(
        index=["scenario_label", "n"],
        columns="method_label",
        values=keep_cols,
        aggfunc="first",
    ).sort_index()
    display(main_table.round(3))
    main_table_path = OUT_DIR / "fci_graph_benchmark_main_table_empirical_mc.csv"
    main_table.to_csv(main_table_path)
    print("Saved main table to:", main_table_path)

    # Compact table for rebuttal, including precision/recall/F1/runtime/features.
    compact_rows = []
    metric_stems = [
        "precision_skeleton", "recall_skeleton", "f1_skeleton",
        "precision_arrow", "recall_arrow", "f1_arrow",
        "runtime_sec", "n_features",
    ]

    for (scenario_label, n), g in main.groupby(["scenario_label", "n"]):
        row = {"scenario": scenario_label, "n": n}
        for _, r in g.iterrows():
            prefix = "revised" if r["method"] == "modified" else "standard"
            for stem in metric_stems:
                row[f"{prefix}_{stem}"] = r[f"{stem}_mean"]
            row[f"{prefix}_n_success"] = r["n_success"]
        compact_rows.append(row)

    compact = pd.DataFrame(compact_rows).sort_values(["scenario", "n"])

    for metric in metric_stems:
        a = f"revised_{metric}"
        b = f"standard_{metric}"
        if a in compact.columns and b in compact.columns:
            compact[f"diff_{metric}_revised_minus_standard"] = compact[a] - compact[b]

    compact_path = OUT_DIR / "fci_graph_benchmark_compact_rebuttal_table_mc.csv"
    compact.to_csv(compact_path, index=False)
    display(compact.round(3))
    print("Saved compact rebuttal table to:", compact_path)


Saved combined summary to: results_fci_graph_benchmark/fci_graph_benchmark_summary_combined_mc.csv


f1_arrow_mean  \
method_label                                      Revised mixed-data KCI   
scenario_label                               n                             
Linear: mixed variables only in group 1      1000                  0.792   
                                             2000                  0.798   
Nonlinear: mixed variables in groups 1 and 2 1000                  0.741   
                                             2000                  0.764   

                                                                          \
method_label                                      Standard KCI + one-hot   
scenario_label                               n                             
Linear: mixed variables only in group 1      1000                  0.598   
                                             2000                  0.603   
Nonlinear: mixed variables in groups 1 and 2 1000                  0.268   
                                             2000                  0.165   

                                                        f1_skeleton_mean  \
method_label                                      Revised mixed-data KCI   
scenario_label                               n                             
Linear: mixed variables only in group 1      1000                  0.927   
                                             2000                  0.931   
Nonlinear: mixed variables in groups 1 and 2 1000                  0.847   
                                             2000                  0.886   

                                                                          \
method_label                                      Standard KCI + one-hot   
scenario_label                               n                             
Linear: mixed variables only in group 1      1000                  0.671   
                                             2000                  0.664   
Nonlinear: mixed variables in groups 1 and 2 1000                  0.323   
                                             2000                  0.260   

                                                         n_features_mean  \
method_label                                      Revised mixed-data KCI   
scenario_label                               n                             
Linear: mixed variables only in group 1      1000                    8.0   
                                             2000                    8.0   
Nonlinear: mixed variables in groups 1 and 2 1000                    8.0   
                                             2000                    8.0   

                                                                          \
method_label                                      Standard KCI + one-hot   
scenario_label                               n                             
Linear: mixed variables only in group 1      1000                   12.0   
                                             2000                   12.0   
Nonlinear: mixed variables in groups 1 and 2 1000                   15.0   
                                             2000                   15.0   

                                                               n_success  \
method_label                                      Revised mixed-data KCI   
scenario_label                               n                             
Linear: mixed variables only in group 1      1000                     50   
                                             2000                     50   
Nonlinear: mixed variables in groups 1 and 2 1000                     50   
                                             2000                     50   

                                                                          \
method_label                                      Standard KCI + one-hot   
scenario_label                               n                             
Linear: mixed variables only in group 1      1000                     50   
                        

Saved main table to: results_fci_graph_benchmark/fci_graph_benchmark_main_table_empirical_mc.csv


,scenario,n,revised_precision_skeleton,revised_recall_skeleton,revised_f1_skeleton,revised_precision_arrow,revised_recall_arrow,revised_f1_arrow,revised_runtime_sec,revised_n_features,...,standard_n_features,standard_n_success,diff_precision_skeleton_revised_minus_standard,diff_recall_skeleton_revised_minus_standard,diff_f1_skeleton_revised_minus_standard,diff_precision_arrow_revised_minus_standard,diff_recall_arrow_revised_minus_standard,diff_f1_arrow_revised_minus_standard,diff_runtime_sec_revised_minus_standard,diff_n_features_revised_minus_standard
0,Linear: mixed variables only in group 1,1000,0.991,0.873,0.927,0.953,0.684,0.792,67.196,8.0,...,12.0,50,0.011,0.362,0.257,0.012,0.241,0.194,-37.093,-4.0
1,Linear: mixed variables only in group 1,2000,0.989,0.880,0.931,0.950,0.691,0.798,328.602,8.0,...,12.0,50,-0.002,0.380,0.267,-0.041,0.256,0.195,-191.962,-4.0
2,Nonlinear: mixed variables in groups 1 and 2,1000,0.994,0.740,0.847,0.905,0.631,0.741,33.290,8.0,...,15.0,50,0.054,0.542,0.524,0.161,0.471,0.473,-119.223,-7.0
3,Nonlinear: mixed variables in groups 1 and 2,2000,0.994,0.800,0.886,0.875,0.681,0.764,217.090,8.0,...,15.0,50,0.001,0.647,0.626,0.396,0.590,0.599,-736.809,-7.0


Saved compact rebuttal table to: results_fci_graph_benchmark/fci_graph_benchmark_compact_rebuttal_table_mc.csv
